# FUNCTIONS: ADVANCED TOPICS
This notebook covers some of the useful features of Python functions

## 1. Returning multiple values
In C++ a function can return at most one value
```c++
T function(args) {
 T val;
 // calculations
 return val;
}
```
where `T` can be any type or class.

**Python functions can return an arbitrary number of values (of arbitrary type combinations).  These are collected in a tuple.**

In [1]:
def xplus(x):
    return x+1, x+2, x+3, x+4

output = xplus(4)
print(type(output))
print(output)

<class 'tuple'>
(5, 6, 7, 8)


### Example: calculating boost parameters

Let's compute simple kinematic information and boost parameters.

For simplcity we assume the momentum of the $\pi$ is along the *x* axis, but you should **TRY TO GENERALIZE THIS EXAMPLE.**

In [2]:
import math as m

m_pi = 0.140 # GeV
p_pi = 1.2   # GeV

def make_p4(mass, p):
    return [m.sqrt(mass**2 + p**2), p, 0, 0] # momentum along the x axis as a list [E, px, py, pz]

p4_pi = make_p4(m_pi, p_pi)
print("pi 4-momentum:", p4_pi)

def boost_params(p4):
    p = m.sqrt(p4[1]**2 + p4[2]**2 + p4[3]**2)
    E = p4[0]
    mass = m.sqrt(E**2 - p**2)
    return p/E, E/mass, p/mass # beta, gamma, beta * gamma (as a tuple)

beta_pi, gamma_pi, betagamma_pi = boost_params(p4_pi)

print(f"pi boost: beta = {beta_pi:.3f}, gamma = {gamma_pi:.3f}, beta*gamma = {betagamma_pi:.3f}")

pi 4-momentum: [1.2081390648431165, 1.2, 0, 0]
pi boost: beta = 0.993, gamma = 8.630, beta*gamma = 8.571


## 2. The `_` variable

If a function returns more values you need to make sure that all of them are redirected to destination variables when calling the function.

#### Example: suppose we only need $\beta$ and $\gamma$ and not $\beta\gamma$.

In [3]:
m_B = 5.279 # GeV
p_B = 0.3   # GeV

beta_B, gamma_B = boost_params(make_p4(m_B, p_B))
print(beta_B)

ValueError: too many values to unpack (expected 2)

In order for the function to work, you are forced to have 3 variables to write to.

This can be tedious because at times you might not need all these returned values, or simply do not care. Fear not: Python has a solution for this as well.

The `_`  is a special variable that can be used for a number of purposes. One of them is to ignore values we do not care about.

Suppose we want to use only $\beta_B$.

In [7]:
beta_B, *_ = boost_params(make_p4(m_B, p_B))

print("B beta:", beta_B, "*_ is ", *_, type(_), _[0], _[1])

B beta: 0.05673740118652557 *_ is  1.0016134628566604 0.05682894487592347 <class 'list'> 1.0016134628566604 0.05682894487592347


In [ ]:
print(_)
print(type(_))
print(len(_))

# In this case `*_` means that 0 or more vales are unpacked and assigned to `_`.

Here `_` is a list of 2 objects and can be used as such.

In [8]:
print(_)
print(_[0], _[1])

[1.0016134628566604, 0.05682894487592347]
1.0016134628566604 0.05682894487592347


#### Similarly

In [9]:
a, b, *_ = xplus(13)
print(a, b, _)

a, b, c, _ = xplus(13)
print(a, b, c, _)

a, b, c, *_ = xplus(13)
print(a, b, c, _)

a, _, c, _ = xplus(13)
print(a, c, _)

a, *_, d = xplus(13)
print(a, d, _)

14 15 [16, 17]
14 15 16 17
14 15 16 [17]
14 16 17
14 17 [15, 16]


#### Watch out, however: only one `*_` is allowed

In [ ]:
*_, c, *_ = xplus(13)

SyntaxError: multiple starred expressions in assignment (3324119802.py, line 1)

#### If the multiple values have special meanings, returning a dictionary should be considered

In [11]:
def boost_dict(p4):
    p = m.sqrt(p4[1]**2 + p4[2]**2 + p4[3]**2)
    E = p4[0]
    mass = m.sqrt(E**2 - p**2)
    return {'beta': p/E, 'gamma': E/mass, 'betagamma': p/mass}

m_mu = 0.106 # GeV
p_mu = 0.020 # GeV

boost_mu = boost_dict(make_p4(m_mu, p_mu))

print(boost_mu)

print("mu beta: ", boost_mu['beta'])

{'beta': 0.1854078591978247, 'gamma': 1.0176442686914566, 'betagamma': 0.18867924528301888}
mu beta:  0.1854078591978247


## 3. Anonymous (`lambda`) functions 
`lambda` functions are a special class of functions that consist of a simple single statement.

Suppose we want to compute `1 + x**2 - x**3` for elements of a list, using a comprehension.

In [12]:
import random as r

def myfunc(x):
    return 1 + x**2 - x**3

def apply_to_list(alist, f):
    return [f(x) for x in alist]

alist = [r.normalvariate(1., 0.3) for i in range(5)]

print(alist)
print(apply_to_list(alist, myfunc))

[0.7397361880258644, 1.0221010894138651, 1.5723814024360756, 0.9630065562011446, 0.8509634067059121]
[1.1424188636997052, 0.976911198822255, -0.41514620614764564, 1.03430704010911, 1.107923167834452]


Function `myfunc()` has really no other use other than when applied to a list. So its name is basically useless.

Further, if we wanted now to apply a new function we would need to define a new useless function.

Rather than definining a standard function with 
```python
def myfunc(x):
    return 1 + x**2 - x**3
```
we can do something more light weight.

We can create functions on the fly which do not have a name. Technically it means the function object does not have the `__name__` attribute [remember: also functions are objects in Python, so they have attributes and `__name__` is one of them].

The solution with a `lambda` function is quite simple.

In [19]:
print(apply_to_list(alist, myfunc))
print(apply_to_list(alist, lambda x: 1 + x**2 - x**3))
print(apply_to_list(alist, lambda x: 1 )) 

[1.1424188636997052, 0.976911198822255, -0.41514620614764564, 1.03430704010911, 1.107923167834452]
[1.1424188636997052, 0.976911198822255, -0.41514620614764564, 1.03430704010911, 1.107923167834452]
[1, 1, 1, 1, 1]


### Sorting lists with lambda functions
A typical use of `lambda` functions is list sorting.

In [22]:
vals = [r.uniform(0., 3.) for i in range(5)]
print("Original data:", vals)
print("Formated data:", [f"{x:0.3f}" for x in vals])

vals.sort()
print("Sorted data:", [f"{x:.3f}" for x in vals])

vals.sort(key=lambda x: m.sin(x))
print("Data sorted by sin with lambda:", [f"{x:.3f}" for x in vals])

vals.sort(key=m.sin)
print("Data sorted by sin without lambda:", [f"{x:.3f}" for x in vals])


vals.sort(key=lambda x: -x)
print("Data sorted by decreasing order with lambda:", [f"{x:.3f}" for x in vals])


Original data: [2.11316387970937, 0.9977586791703668, 2.714692089160142, 2.5461070565043435, 2.3121326334969634]
Formated data: ['2.113', '0.998', '2.715', '2.546', '2.312']
Sorted data: ['0.998', '2.113', '2.312', '2.546', '2.715']
Data sorted by sin with lambda: ['2.715', '2.546', '2.312', '0.998', '2.113']
Data sorted by sin without lambda: ['2.715', '2.546', '2.312', '0.998', '2.113']
Data sorted by decreasing order with lambda: ['2.715', '2.546', '2.312', '2.113', '0.998']


In [27]:
vals.sort(key=lambda x: 1 + x**2 - x**3)
print("Data sorted by 1+x**2-x**3 with lambda, \nso the first element will have the smaller value of the function:\n", [f"{x:.3f}" for x in vals])

Data sorted by 1+x**2-x**3 with lambda, 
so the first element will have the smaller value of the function:
 ['2.715', '2.546', '2.312', '2.113', '0.998']


In [29]:
def apply_to_list_of_tuples(alist, f):
    return [f(v1,v2) for v1,v2 in alist]

vals2 = [(0,1), (1,10), (-9, 5.5)]

vals3 = apply_to_list_of_tuples(vals2, lambda x,y: 1 + x**2 - y**3)
print(vals3)

[0, -998, -84.375]


As an additonal use, we can sort the numbers based on unique numerals appearing in the number itself

In [30]:
vals

[2.714692089160142,
 2.5461070565043435,
 2.3121326334969634,
 2.11316387970937,
 0.9977586791703668]

In [31]:
print([f"{x:0.3f}" for x in vals])

new_vals = [set(f"{x:.3f}") for x in vals]
print(new_vals)

vals.sort(key=lambda x: len(set(f"{x:.3f}")))
print([f"{x:.3f}" for x in vals])

['2.715', '2.546', '2.312', '2.113', '0.998']
[{'5', '.', '7', '2', '1'}, {'5', '.', '4', '6', '2'}, {'3', '.', '1', '2'}, {'3', '.', '1', '2'}, {'0', '.', '9', '8'}]
['2.312', '2.113', '0.998', '2.715', '2.546']


## 4. Functions with arbitrary number of arguments

As seen for example with the `print()` function, functions can have a variable number of arguments. The same behaviour can easily be defined for any custom defined function for both **positional and keyword arguments**.

### Positional arguments

Additional arguments are taken via the special `*arg` argument which is a **tuple** of additional positional arguments.

In [33]:
def myfunc(a, *arg):
    print(f"Positional arguments: {a} {arg}")
    if len(arg):
        for x in arg:
            print(f'[{x}]\t')
        print(type(arg), '\n')

In [34]:
myfunc()

TypeError: myfunc() missing 1 required positional argument: 'a'

In [35]:
myfunc(1.1)

Positional arguments: 1.1 ()


In [36]:
myfunc(1.1, )

Positional arguments: 1.1 ()


In [37]:
myfunc('hello')

Positional arguments: hello ()


In [38]:
myfunc(-0.2, 0.3, 'hello')

Positional arguments: -0.2 (0.3, 'hello')
[0.3]	
[hello]	
<class 'tuple'> 



In [39]:
myfunc(-0.2, 0.3, 'hello', 'goodbye', -2, 100, )

Positional arguments: -0.2 (0.3, 'hello', 'goodbye', -2, 100)
[0.3]	
[hello]	
[goodbye]	
[-2]	
[100]	
<class 'tuple'> 



### Keyword arguments

For optional keyword arguments, the `**karg` feature is used.  This is a **dictionary**.

In [41]:
def myfunc2(a, mu=0.0, sig=0.1, **karg):
    print(f"a: {a}")
    print(f"Keyword arguments: {mu} {sig} {karg}")
    if len(karg):
        for x in karg:
            print(f'[{x}]\t')
        print(type(karg), '\n')

In [42]:
myfunc2()

TypeError: myfunc2() missing 1 required positional argument: 'a'

In [43]:
myfunc2(0.11111)

a: 0.11111
Keyword arguments: 0.0 0.1 {}


In [44]:
myfunc2(0.3, sig=0.5)

a: 0.3
Keyword arguments: 0.0 0.5 {}


In [45]:
myfunc2(0.3, color='red')

a: 0.3
Keyword arguments: 0.0 0.1 {'color': 'red'}
[color]	
<class 'dict'> 



In [46]:
myfunc2(2., color='red', mu=0.6)

a: 2.0
Keyword arguments: 0.6 0.1 {'color': 'red'}
[color]	
<class 'dict'> 



The additional keyword arguments are stored as a dictionary.

In [47]:
def myfunc3(a,mu=0.0, sig=0.1, **karg):
    print(f"a: {a}")
    print(f"Keyword arguments: {mu} {sig} {karg}")
    if len(karg):
        for x in karg.keys():
            print(f'[{x} = {karg[x]}]\t')
        print(type(karg), '\n')
        
myfunc3(0.1)

a: 0.1
Keyword arguments: 0.0 0.1 {}


In [48]:
myfunc3(0.3, color='red', mu=0.6)

a: 0.3
Keyword arguments: 0.6 0.1 {'color': 'red'}
[color = red]	
<class 'dict'> 



### Combine both positional and keyword arguments for the most generic function

In [51]:
def myfunc4(a, *arg, mu=0, sig=1, **karg):
    print("Myfunc4 called")
    print(f"Positional:  a: {a}.     Optional: {arg}")
    if len(arg):
        for x in arg:
            print(f'[{x}]\t')
        print(type(arg), '\n')
    print(f"keyword: {mu} {sig} {karg}")    
    if len(karg):
        for x in karg.keys():
            print(f'[{x} = {karg[x]}]\t')
        print(type(karg), '\n')
    
    try:
        print(f"{karg['color']} is my favourite color")  #@markdown Note the careful placement of single/double quotes
    except KeyError:
        pass
    
myfunc4(-0.1)

myfunc4(0.3, 'x', 'y', 0.9, color='red', mu=0.6, thick=1.1, fill='true')

Myfunc4 called
Positional:  a: -0.1.     Optional: ()
keyword: 0 1 {}
Myfunc4 called
Positional:  a: 0.3.     Optional: ('x', 'y', 0.9)
[x]	
[y]	
[0.9]	
<class 'tuple'> 

keyword: 0.6 1 {'color': 'red', 'thick': 1.1, 'fill': 'true'}
[color = red]	
[thick = 1.1]	
[fill = true]	
<class 'dict'> 

red is my favourite color


In [53]:
myfunc4(78)

Myfunc4 called
Positional:  a: 78.     Optional: ()
keyword: 0 1 {}


In [ ]:
myfunc4(-0.1, 10.1)
def myfunc4bis(a, mu=0, sig=1, **karg):
    print("Myfunc4 called")
    print(f"keyword: {mu} {sig} {karg}, note how mu changed!")    
    if len(karg):
        for x in karg.keys():
            print(f'[{x} = {karg[x]}]\t')
        print(type(karg), '\n')
    
    try:
        print(f"{karg['color']} is my favourite color")  #@markdown Note the careful placement of single/double quotes
    except KeyError:
        pass
myfunc4bis(-0.1, 10.1)

Myfunc4 called
Positional:  a: -0.1.     Optional: (10.1,)
[10.1]	
<class 'tuple'> 

keyword: 0 1 {}
Myfunc4 called
keyword: 10.1 1 {}, note how mu change!


In [61]:
myfunc4(-0.1, mu=10.1)

Myfunc4 called
Positional:  a: -0.1.     Optional: ()
keyword: 10.1 1 {}


In [62]:
myfunc4(0.3, 'x', 'y', 0.9, color='red', mu=0.6, thick=1.1, fill='true')

Myfunc4 called
Positional:  a: 0.3.     Optional: ('x', 'y', 0.9)
[x]	
[y]	
[0.9]	
<class 'tuple'> 

keyword: 0.6 1 {'color': 'red', 'thick': 1.1, 'fill': 'true'}
[color = red]	
[thick = 1.1]	
[fill = true]	
<class 'dict'> 

red is my favourite color


# COMMAND LINE ARGUMENTS

Since we covered function arguments, it is only natural to wonder how to provide arguments to a Python script being run from command line.

The `sys` module grants easy access to command line arguments as a list. An example is in `examples/Python/cmd_line_args.py`, which reads:

```Python
import sys, os

print("Running " + __file__)

print("Running " + os.path.basename(__file__))

print(f"Program called with {len(sys.argv)} arguments")

for a in sys.argv:
    print(a)
```

Run this script in a terminal to see what happens.

# READY FOR `examples/Python/4-AnimatedPlots.ipynb`!